In [ ]:
import pymysql
import urllib.parse
import datetime
import decimal
import pandas as pd
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
from bson import json_util

### Connecting to MySQL to extract source

In [ ]:
mysql_conn = pymysql.connect(
    host='INSTITUTION',
    user='USERNAME',
    password='PASSWORD',
    database='emr'
)
mysql_cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)

with mysql_conn.cursor() as cursor:
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
    print("MySQL version:", version[0])

MySQL version: 8.4.5


### Connecting to MongoDb to load target

In [ ]:
import urllib.parse
username ='USERNAME'
password = urllib.parse.quote_plus("PASSWORD")
host = "HOST"
port = 27017
auth_db = "admin"# or the database where the user is defined

# Format the connection string
uri = f"mongodb://{username}:{password}@{host}:{port}/?authSource={auth_db}"
mongo_client = MongoClient(uri)
mongo_db = mongo_client['client_DB']
mongo_client.admin.command('ping')# Trigger connection
print(" Successfully connected to MongoDB with authentication.")
print("Available databases:", mongo_client.list_database_names())

 Successfully connected to MongoDB with authentication.
Available databases: ['etl', 'mam1779', 'sakila']


In [ ]:
mongo_db = mongo_client["client_DB"];

Run helper functions to sanitize data for MongoDB. I want to convert MySQL returned Python objects into types that MondoDB can store safetly. Python can return decimal objects that MongoDB cannot serialize by default. Also, MySQL date values may come through as datetime.date, so we want to convert it to datetime.datetime as mongo stoes dates as such.

In [ ]:
import datetime, decimal

def sanitize_for_mongo(doc):
    if isinstance(doc, dict):
        return {k: sanitize_for_mongo(v) for k, v in doc.items()}
    if isinstance(doc, list):
        return [sanitize_for_mongo(x) for x in doc]
    if isinstance(doc, decimal.Decimal):
        return float(doc)
    if isinstance(doc, datetime.date) and not isinstance(doc, datetime.datetime):
        return datetime.datetime.combine(doc, datetime.time())
    return doc

Running the MySQL `mysql_fetchall` function in order to standardize SQL execution and fetch results as list of dict rowa

In [ ]:
def mysql_fetchall(sql, params=None):
    mysql_cursor.execute(sql, params or ())
    return mysql_cursor.fetchall()

Now, I am resetting MongoDB Target Collections in order to clear the target collections preventing dupes and ensures each run starts clean because users may run through this notebook multiple times during testing and development, so this helps keep things smooth.

In [ ]:
for coll in ["patients", "providers", "diagnoses", "labs", "clinical_procedures", "symptoms"]:
    mongo_db[coll].delete_many({})

print("Cleared target collections to avoid duplicate inserts.")

Cleared target collections to avoid duplicate inserts.


Loading Reference/Catalog Collections. I am running this code in order to load 'shared' tables first, the provider, diagnosis, lab, clinical_procedures, and symptom, so I can reuse them and reference them later.

Building  the This order matters because the patient documents embed visits, and each visit stores a provider ref using `ObjectID()` called provider_ref. Requiring inserting the providers first

In [ ]:
providers = [sanitize_for_mongo(x) for x in mysql_fetchall('SELECT * FROM provider')]

if providers:
    mongo_db.providers.insert_many(providers)

# bulding provider_old_map to map provider_id -> mongoDb _id, so mysql integer to object id
provider_oid_map = {
    p["provider_id"]: p["_id"]
    for p in mongo_db.providers.find({}, {"provider_id": 1})
}
print("Loaded providers:", mongo_db.providers.count_documents({}))


Loaded providers: 50


In [ ]:
# Load diagnoses catalog
diagnoses = [sanitize_for_mongo(x) for x in mysql_fetchall("SELECT * FROM diagnosis")]
if diagnoses:
    mongo_db.diagnoses.insert_many(diagnoses)
print("Loaded diagnoses:", mongo_db.diagnoses.count_documents({}))

Loaded diagnoses: 500


In [ ]:
# Load labs catalog
labs = [sanitize_for_mongo(x) for x in mysql_fetchall("SELECT * FROM lab")]
if labs:
    mongo_db.labs.insert_many(labs)
print("Loaded labs:", mongo_db.labs.count_documents({}))

Loaded labs: 500


In [ ]:
# Load clinical procedures catalog
procedures = [sanitize_for_mongo(x) for x in mysql_fetchall("SELECT * FROM clinical_procedures")]
if procedures:
    mongo_db.clinical_procedures.insert_many(procedures)
print("Loaded clinical_procedures:", mongo_db.clinical_procedures.count_documents({}))

Loaded clinical_procedures: 500


In [ ]:
# Load symptoms catalog 
symptoms = [sanitize_for_mongo(x) for x in mysql_fetchall("SELECT * FROM symptom")]
if symptoms:
    mongo_db.symptoms.insert_many(symptoms)
print("Loaded symptoms:", mongo_db.symptoms.count_documents({}))

Loaded symptoms: 500


Building embedded visit clincial array. In MySQL diagnoses, labs, procedures, and symptoms are many to many with vist implemented using juncion tables of visit_diagnosis, visit_lab, visit_procedure, and visit_symptom. For MongoDB we embed these details directly inside each visit documant to produce a self-contained visit history.

In [ ]:
def get_visit_diagnoses(visit_id):
    # Extract diagnoses for one visit by joining visit_diagnosis to diagnosis

    sql = """
    SELECT d.*
    FROM visit_diagnosis vd
    JOIN diagnosis d ON d.diagnosis_id = vd.diagnosis_id
    WHERE vd.visit_id = %s
    """
    return [sanitize_for_mongo(x) for x in mysql_fetchall(sql, (visit_id,))]

In [ ]:
def get_visit_labs(visit_id):
    # Extract diagnoses for one visit by joining visit_lab -> lab
    sql = """
    SELECT l.*
    FROM visit_lab vl
    JOIN lab l ON l.lab_id = vl.lab_id
    WHERE vl.visit_id = %s
    """
    return [sanitize_for_mongo(x) for x in mysql_fetchall(sql, (visit_id,))]

In [ ]:
def get_visit_procedures(visit_id):
    # Extract procedures for one visit by joining visit_procedure to clinical_procedures
    sql = """
    SELECT cp.*
    FROM visit_procedure vp
    JOIN clinical_procedures cp ON cp.procedure_id = vp.procedure_id
    WHERE vp.visit_id = %s
    """
    return [sanitize_for_mongo(x) for x in mysql_fetchall(sql, (visit_id,))]

In [ ]:
def get_visit_symptoms(visit_id):
    # Extract symptoms for one visit by joining visit_symptom to symptom
    sql = """
    SELECT s.*
    FROM visit_symptom vs
    JOIN symptom s ON s.symptom_id = vs.symptom_id
    WHERE vs.visit_id = %s
    """
    return [sanitize_for_mongo(x) for x in mysql_fetchall(sql, (visit_id,))]

Build embedded visits for a patient document. So, for each patient pull all visits and embed them inside the patient document. Each embedded visit includes: visit_id, visit_date, provider reference(provider_id + privder_red ObjectId), embedded arrays of diagnoses, lab, procedures, and symptoms

In [ ]:
def get_patient_visits(patient_id):
    # Extract all visits for this patient from MySQL
    visits = [
        sanitize_for_mongo(v)
        for v in mysql_fetchall("SELECT * FROM visit WHERE patient_id = %s", (patient_id,))
    ]

    for v in visits:
        visit_id = v["visit_id"]
        prov_id = v.get("provider_id")

        # Provider is stored as a referenced collection in MongoDB.
        # We attach both: provider_id (original MySQL identifier) and provider_ref (MongoDB ObjectId link to providers collection)
        v["provider"] = {
            "provider_id": prov_id,
            "provider_ref": provider_oid_map.get(prov_id)
        }

        # Embed clinical data inside the visit so the visit record is self-contained.
        v["diagnoses"] = get_visit_diagnoses(visit_id)
        v["labs"] = get_visit_labs(visit_id)
        v["procedures"] = get_visit_procedures(visit_id)
        v["symptoms"] = get_visit_symptoms(visit_id)

    return visits

Inserting patient documents into MongoDB

In [ ]:
patients = [sanitize_for_mongo(p) for p in mysql_fetchall("SELECT * FROM patient")]

patient_docs = []
for p in patients:
    # Attach embedded visits array for this patient (including visit_date + clinical arrays)
    p["visits"] = get_patient_visits(p["patient_id"])
    patient_docs.append(p)

# Insert into MongoDB if there is data to load
if patient_docs:
    mongo_db.patients.insert_many(patient_docs)

print("Loaded patients:", mongo_db.patients.count_documents({}))

Loaded patients: 50


Verifying the ETL producded patient documents with embedded visits.

In [ ]:
sample = mongo_db.patients.find_one(
    {"visits.0": {"$exists": True}},
    {"_id": 0}
)
sample

{'patient_id': 1,
 'first_name': 'Megan',
 'last_name': 'Chang',
 'dob': datetime.datetime(1991, 7, 7, 0, 0),
 'gender': 'Female',
 'visits': [{'visit_id': 0,
   'patient_id': 1,
   'provider_id': 21,
   'visit_date': datetime.datetime(2024, 3, 26, 0, 0),
   'provider': {'provider_id': 21,
    'provider_ref': ObjectId('699e053961f9b683c028b8c6')},
   'diagnoses': [],
   'labs': [],
   'procedures': [],
   'symptoms': []},
  {'visit_id': 1,
   'patient_id': 1,
   'provider_id': 6,
   'visit_date': datetime.datetime(2024, 3, 27, 0, 0),
   'provider': {'provider_id': 6,
    'provider_ref': ObjectId('699e053961f9b683c028b8b7')},
   'diagnoses': [{'diagnosis_id': 91,
     'name': 'Psoriasis vulgaris',
     'icd10_code': 'L40.0'},
    {'diagnosis_id': 104, 'name': 'Psoriasis vulgaris', 'icd10_code': 'L40.0'},
    {'diagnosis_id': 142,
     'name': 'Migraine, unspecified, not intractable, without status migrainosus',
     'icd10_code': 'G43.909'},
    {'diagnosis_id': 223,
     'name': 'Acute